# Identifiability and Training-Evaluation Mismatch — Level-Set Sampling (Read-Only Analysis)

> **Read-only notebook**: displays artifacts loaded from `results/exp_r21_r22_levelset/`; performs no computation or file writes itself.
> Artifacts are generated by `033_exp_r21_r22_levelset.py` (data source = the frozen Experiment 0 baseline artifacts, test-fold `grid_demands` across 3 seeds + graph cache; pure CPU post-processing, no retraining).

## What This Experiment Addresses

- **Identifiability**: *"many different spatial allocations can reproduce the same aggregate land-use mixture … justify why $w_{sa}$ is identifiable"* — we uniformly sample alternative allocations on the **exact level set** of the land-use reconstruction loss, quantifying the size of the solution space (nullspace dimension) and the RMSE percentile of the learned solution within it.
- **Training-evaluation mismatch**: *"a redistribution that barely changes $L_{landuse}$ may substantially change substation RMSE"* — we directly measure how much substation RMSE can change under a redistribution that leaves the loss **exactly unchanged** (within the level set), and compare the magnitude to the paper's antagonism gap ($+1.93$ MVA).

## Loss Implementation Verification (Precondition, Verified Programmatically per ITL3 Region in `033_exp_r21_r22_levelset.py`)

`LandusePredictionLoss` reconstructs a **category one-hot aggregate mixture**: each agent's unique category = $\arg\max$(lu\_*\_prop), and the predicted mixture for each ITL3 source $= \sum_{a: cat(a)=c} w_{sa}$ (row-normalized, then KL-divergence against the normalized `{cat}_percent` target). The aggregation domain is ITL3 (star graph); it **does not include** the GVA component. The baseline configuration's objective function consists **only** of `landuse_prediction_loss` — so this level set is a **complete characterization** of the baseline training objective's solution space (the key advantage of this design).

## Two Documented Deviations (Findings Arising from the Loss Structure)

- **Deviation 1**: In the constraint matrix $A = [\text{5 category one-hot rows}; \mathbf{1}^\top]$, the all-ones row equals the sum of the indicator rows → $\mathrm{rank}(A) = $ the number of categories actually present $\le 5$, **not the originally expected 6**; nullspace dimension $= n - \mathrm{rank}(A)$ (rank computed and verified directly for each ITL3 region).
- **Deviation 2**: The one-hot structure implies the level set is a **product of per-category scaled simplices**, which admits an exact i.i.d. uniform sampler (block-wise Dirichlet(1,…,1)). The primary readout uses `dirichlet_exact` (provably uniform, no MCMC burn-in risk); hit-and-run (the originally specified sampler) is retained in full as a control — under tens of thousands of steps in this high-dimensional setting, MCMC samples necessarily remain confined to a lower-dimensional slice, expected to manifest as an artificially narrow spread; the consistency diagnostic is saved to disk.

In [ ]:
# Load artifacts (read-only)
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# CJK font fallback (Windows)
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

EXP_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = EXP_DIR / 'results' / 'exp_r21_r22_levelset'

with open(OUT / 'identifiability_metrics.json', encoding='utf-8') as f:
    R = json.load(f)
SUMMARY = pd.read_csv(OUT / 'levelset_samples_summary.csv')
NS = pd.read_csv(OUT / 'nullspace_dims.csv')
NPZ = np.load(OUT / 'rmse_samples.npz')

META = R['meta']
print('Artifacts:', OUT)
print(f"Config: {META['config']} × seeds {META['seeds']} × {META['n_locations']} regions = {META['n_units']} units")
print(f"K = {META['K']} per unit·base·sampler; rng_seed = {META['rng_seed']}; primary sampler = {META['primary_sampler']}")
print(f"Generated at {META['timestamp']}  numpy {META['numpy']}")
print()
print('── Precondition assertions (expected / actual / status) ──')
pre_rows = []
for key, item in R['preconditions'].items():
    pre_rows.append({'precondition': key, 'description': item['description'],
                     'expected': json.dumps(item['expected'], ensure_ascii=False),
                     'actual': json.dumps(item['actual'], ensure_ascii=False),
                     'status': item['status']})
with pd.option_context('display.max_colwidth', 110):
    display(pd.DataFrame(pre_rows))

## 1. Nullspace Dimension (Solution Space Size)

Each ITL3 region has only $\mathrm{rank}(A)$ (= number of categories actually present, empirically always $\le 5$) binding constraints — **each constraint pins down 1 dimension, leaving $n - \mathrm{rank}(A)$ dimensions completely free**. The reviewer's intuition is exactly right in terms of dimensionality: the level set is a polytope with hundreds to tens of thousands of dimensions.

In [ ]:
print('Rank distribution (all ITL3 regions):', R['nullspace']['rank_distribution'],
      ' [originally expected rank=6, empirically ≤5: the all-ones row is linearly dependent on the one-hot rows — Deviation 1]')
print(f"Nullspace dimension range: [{R['nullspace']['nullspace_dim_min']}, {R['nullspace']['nullspace_dim_max']}], "
      f"median {R['nullspace']['nullspace_dim_median']:.0f} (across {R['nullspace']['n_itl3_total']} ITL3 regions)")

per_loc = NS.groupby('location').agg(
    n_itl3=('itl3', 'count'), n_agents=('n_agents', 'sum'),
    rank_min=('rank_A', 'min'), rank_max=('rank_A', 'max'),
    nullspace_dim_min=('nullspace_dim', 'min'),
    nullspace_dim_max=('nullspace_dim', 'max'),
    nullspace_dim_total=('nullspace_dim', 'sum'))
display(per_loc)

## 2. Level-Set RMSE Distribution and the Percentile of the Learned Solution

Violin plots = the substation RMSE distribution of 200 uniform samples on the level set (`dirichlet_exact`, exactly matching the loss value of the learned solution element-wise); red dot = the learned solution (GNN); blue triangle = the level-set barycenter (block-wise uniform allocation).

Key to interpretation: uniform measure **concentrates** on high-dimensional polytopes — RMSE values of typical members cluster closely together (narrow violins). If the learned solution deviates significantly from the typical band, this indicates that the training dynamics selected an **atypical member** of the level set (one the loss itself cannot distinguish from the others).

In [ ]:
SEED_SHOW = 42
locs = list(SUMMARY['location'].unique())
fig, axes = plt.subplots(2, 8, figsize=(22, 7), sharey=False)
for ax, loc in zip(axes.ravel(), locs):
    arr = NPZ[f'{SEED_SHOW}|{loc}|gnn|dirichlet_exact']
    row = SUMMARY[(SUMMARY['seed'] == SEED_SHOW) & (SUMMARY['location'] == loc)
                  & (SUMMARY['base'] == 'gnn') & (SUMMARY['sampler'] == 'dirichlet_exact')].iloc[0]
    ax.violinplot(arr, positions=[0], widths=0.8, showmedians=True)
    ax.scatter([0], [row['rmse_base_point']], color='crimson', zorder=5, s=48,
               label=f"Learned solution (P{row['percentile_base_point']:.0f})")
    ax.scatter([0], [row['rmse_barycenter']], color='steelblue', marker='^', zorder=5, s=40,
               label='Barycenter')
    ax.set_title(f"{loc}\nPercentile {row['percentile_base_point']:.1f}%", fontsize=10)
    ax.set_xticks([])
    ax.tick_params(labelsize=8)
axes[0, 0].set_ylabel('Substation RMSE (MVA)')
axes[1, 0].set_ylabel('Substation RMSE (MVA)')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', fontsize=10)
fig.suptitle(f'Level-Set RMSE Distribution (seed {SEED_SHOW}, GNN base, exact uniform sampling, K=200)', fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

In [ ]:
# Learned-solution percentile summary (48 units) + Uniform control
gnn = SUMMARY[(SUMMARY['base'] == 'gnn') & (SUMMARY['sampler'] == 'dirichlet_exact')]
uni = SUMMARY[(SUMMARY['base'] == 'uniform') & (SUMMARY['sampler'] == 'dirichlet_exact')]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(gnn['percentile_base_point'], bins=np.arange(0, 105, 5),
             color='crimson', alpha=0.7)
axes[0].set_title('Percentile of the Learned Solution Within Its Level-Set Distribution (48 units, GNN base)')
axes[0].set_xlabel('Percentile (%)'); axes[0].set_ylabel('Number of units')
axes[1].hist(uni['percentile_base_point'], bins=np.arange(0, 105, 5),
             color='gray', alpha=0.7)
axes[1].set_title('Uniform Base-Point Percentile (control; the base point is the level-set barycenter)')
axes[1].set_xlabel('Percentile (%)')
fig.tight_layout(); plt.show()

r21 = R['r21_identifiability']
print('Identifiability readout (exact sampler):')
print(f"  Learned-solution percentile  median across regions (seed-averaged) = {r21['learned_percentile']['median_across_locations_seed_mean']:.2f}%")
print(f"              median across 48 units = {r21['learned_percentile']['median_across_48_units']:.2f}%  "
      f"IQR = {r21['learned_percentile']['iqr_across_48_units']}")
print(f"  Median barycenter percentile = {r21['barycenter_percentile_median']:.2f}%")
print(f"  Verdict field (rule-generated): {r21['verdict']}")
print(f"  Rule: {r21['verdict_rule']}")
print()
print(f"Uniform control: median percentile = {R['uniform_control']['percentile']['median_across_locations_seed_mean']:.2f}%")

## 3. RMSE Freedom Under a Fixed Loss vs. the Antagonism Gap

Two complementary framings:
1. **Between typical members** (P95−P5): the RMSE freedom among typical level-set members under the uniform measure;
2. **Learned solution → typical member** (|learned − P50|): the learned solution is itself a level-set member — replacing it with a typical member is a redistribution with **exactly zero change in loss**, and the resulting RMSE shift is an **empirically demonstrated lower bound** for the claim under test.

In [ ]:
r22 = R['r22_evaluation_freedom']
delta_np = r22['antagonism_delta_np_vs_base']

fig, ax = plt.subplots(figsize=(12, 4.5))
x = np.arange(len(gnn))
order = gnn.sort_values(['location', 'seed']).reset_index(drop=True)
ax.scatter(x, order['rmse_p95_p5'], label='P95−P5 (between typical members)', color='steelblue', s=28)
ax.scatter(x, order['learned_minus_p50'].abs(), label='|learned solution − P50| (learned → typical)',
           color='crimson', s=28, marker='D')
ax.axhline(abs(delta_np), color='black', ls='--',
           label=f'Antagonism gap |ΔNP| = {abs(delta_np):.3f} MVA (reference value from the paper)')
ax.set_xticks(x[::3]); ax.set_xticklabels(order['location'][::3], rotation=45, fontsize=8)
ax.set_ylabel('RMSE change (MVA)'); ax.set_yscale('log')
ax.set_title("RMSE Freedom Within the Level Set vs. the Paper's Antagonism Gap (48 units, GNN base, exact sampler)")
ax.legend(); fig.tight_layout(); plt.show()

print('Training-evaluation mismatch readout (median across 48 units):')
print(f"  Level-set P95−P5 = {r22['levelset_rmse_p95_p5_median']:.4f} MVA, range = {r22['levelset_rmse_range_median']:.4f} MVA")
print(f"  |learned solution − P50| = {r22['learned_minus_p50_abs_median']:.4f} MVA (signed median {r22['learned_minus_p50_median']:+.4f})")
print(f"  Antagonism gap ΔNP = {delta_np:+.4f} (ΔN = {r22['delta_n_vs_base']:+.4f}, ΔP = {r22['delta_p_vs_base']:+.4f})")
print(f"  Ratio: P95−P5/|ΔNP| = {r22['ratio_p95p5_to_antagonism']:.3f} → {r22['verdict']}")
print(f"        |learned−P50|/|ΔNP| = {r22['ratio_learned_p50_to_antagonism']:.3f} → {r22['verdict_learned_to_typical']}")
print()
print('Interpretation:', r22['reading_note'])

## 4. Hit-and-Run Control (Consistency Diagnostic for the Originally Specified Sampler)

The level set is a polytope with thousands to tens of thousands of dimensions: samples from a 9,000-step MCMC chain necessarily fall within a ≤9,000-dimensional slice, which in theory must **underestimate the spread** (IQR ratio < 1). This is exactly why the primary readout switched to the exact i.i.d. uniform sampler (Deviation 2); the HR results are retained in full to show whether the originally specified sampler agrees with the exact sampler on the median, and by how much their spreads differ.

In [ ]:
hr = R['hr_vs_exact_agreement']
if hr is None:
    print('This run used --skip-hr; no HR control data available.')
else:
    hr_df = pd.DataFrame(hr['per_unit'])
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    axes[0].hist(hr_df['ks_stat'], bins=20, color='seagreen', alpha=0.75)
    axes[0].set_title('Two-Sample KS Statistic'); axes[0].set_xlabel('KS')
    axes[1].hist(hr_df['median_diff'], bins=20, color='seagreen', alpha=0.75)
    axes[1].set_title('Median Difference (HR − Exact)'); axes[1].set_xlabel('MVA')
    axes[2].hist(hr_df['iqr_ratio_hr_over_exact'], bins=20, color='seagreen', alpha=0.75)
    axes[2].axvline(1.0, color='black', ls='--')
    axes[2].set_title('IQR Ratio (HR / Exact) — <1 indicates underestimated spread')
    fig.tight_layout(); plt.show()
    print(f"Median KS = {hr['ks_stat_median']:.3f} (max {hr['ks_stat_max']:.3f}); "
          f"median of median differences = {hr['median_diff_median']:+.4f} MVA; "
          f"median IQR ratio = {hr['iqr_ratio_median']:.3f} (min {hr['iqr_ratio_min']:.3f})")
    print('Note:', hr['note'])

## 5. Conclusions (All verdict fields are rule-generated from the numeric results; see `identifiability_metrics.json`)

In [ ]:
print('═' * 70)
med = R['r21_identifiability']['learned_percentile']['median_across_locations_seed_mean']
print(f"Identifiability: {R['r21_identifiability']['verdict']}  (median learned-solution percentile {med:.1f}%)")
if med <= 25:
    print('   ', R['r21_identifiability']['interpretation_low'])
else:
    # Percentile is mid-range or high: the loss does not pin down the learned solution in either
    # case (mid-range = the learned solution is indistinguishable from a random member;
    # high = the member selected by training dynamics is worse than a typical member on the
    # held-out region) -- both cases indicate the solution is chosen by inductive bias rather
    # than the loss itself; the paired design of the antagonism analysis is unaffected
    print('   ', R['r21_identifiability']['interpretation_mid'])
print()
print('Training-evaluation mismatch:')
print('  Freedom between typical members       :', R['r22_evaluation_freedom']['verdict'])
print('  Shift from learned solution to typical :', R['r22_evaluation_freedom']['verdict_learned_to_typical'])
print('═' * 70)
print()
print('Documented deviations (deviations_registry):')
for d in R['deviations_registry']:
    print(' •', d)